# Day 4: PHI Governance & De-Identification Engine

This notebook demonstrates applying Format-Preserving Encryption (FPE) for SSN/MRN, deterministic date-shifting, and free-text Safe Harbor 18 redaction on synthetic data.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, date_add
from pyspark.sql.types import StringType
import hashlib
import re
from datetime import date

spark = SparkSession.builder \
    .appName("Day04_DeIdentification") \
    .getOrCreate()

print("Spark session initialized.")

In [ ]:
def deterministic_fpe_mock(val: str) -> str:
    if not val: return val
    hashed = hashlib.sha256(val.encode()).hexdigest()
    return hashed[:len(val)] if len(val) <= 64 else hashed

fpe_udf = udf(deterministic_fpe_mock, StringType())

def redact_free_text_mock(text: str) -> str:
    if not text: return text
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[REDACTED_SSN]', text)
    text = re.sub(r'\b\d{3}-\d{3}-\d{4}\b', '[REDACTED_PHONE]', text)
    return text

redact_udf = udf(redact_free_text_mock, StringType())

In [ ]:
data = [
    ("123-45-6789", "MRN001", date(1980, 5, 15), "Patient SSN is 123-45-6789 and phone is 555-123-4567."),
    ("987-65-4321", "MRN002", date(1992, 10, 20), "No PHI in this note.")
]
columns = ["ssn", "mrn", "birth_date", "notes"]
df_raw = spark.createDataFrame(data, columns)

print("Raw DataFrame:")
df_raw.show(truncate=False)

In [ ]:
df_deid = df_raw \
    .withColumn("ssn_fpe", fpe_udf(col("ssn"))) \
    .withColumn("mrn_fpe", fpe_udf(col("mrn"))) \
    .withColumn("birth_date_shifted", date_add(col("birth_date"), -30)) \
    .withColumn("notes_redacted", redact_udf(col("notes"))) \
    .drop("ssn", "mrn", "birth_date", "notes")

print("De-identified DataFrame:")
df_deid.show(truncate=False)